# Progetto Statistical Learning

considero i df trovato su kaggle e restringo l'analisi all'Italia e ai vini contententi principalmente Merlot

In [9]:
#libraries
import pandas as pd
import numpy as np


In [10]:
#dataset
data=pd.read_csv("cleansingWine.csv")

In [180]:
data=pd.DataFrame(data)

In [181]:
data.head(5)

,Unnamed: 0,id,name,producer,nation,local1,local2,local3,local4,varieties1,...,use,abv,degree,sweet,acidity,body,tannin,price,year,ml
0,0,137197,Altair,Altair,Chile,Rapel Valley,NaN,NaN,NaN,Cabernet Sauvignon,...,Table,14~15,17~19,SWEET1,ACIDITY4,BODY5,TANNIN4,220000,2014,750
1,1,137198,"Altair, Sideral",Altair,Chile,Rapel Valley,NaN,NaN,NaN,Cabernet Sauvignon,...,Table,14~15,16~18,SWEET1,ACIDITY3,BODY4,TANNIN4,110000,2016,750
2,2,137199,Baron du Val Red,Baron du Val,France,NaN,NaN,NaN,NaN,Carignan,...,Table,11~12,15~17,SWEET2,ACIDITY3,BODY2,TANNIN2,0,0,750
3,3,137200,Baron du Val White,Baron du Val,France,NaN,NaN,NaN,NaN,Carignan,...,Table,11~12,9~11,SWEET1,ACIDITY3,BODY2,TANNIN1,0,0,750
4,4,137201,"Benziger, Cabernet Sauvignon",Benziger,USA,California,NaN,NaN,NaN,Cabernet Sauvignon,...,Table,13~14,17~19,SWEET1,ACIDITY3,BODY3,TANNIN4,0,2003,750


In [182]:
Italy=data[data.nation=="Italy"]

In [183]:
Italy_Merlot=Italy[Italy.varieties1=="Merlot"]

In [203]:
Italy_Merlot.iloc[23:27,1:34]

,id,name,producer,nation,local1,local2,local3,local4,varieties1,varieties2,...,use,abv,degree,sweet,acidity,body,tannin,price,year,ml
2207,140831,"Frescobaldi, Lamaione",Frescobaldi,Italy,Toscana,NaN,NaN,NaN,Merlot,NaN,...,Table,14~15,17~20,SWEET1,ACIDITY3,BODY3,TANNIN3,125000,2014,750
2208,140832,"Luce Della Vite, Lucente",Luce Della Vite,Italy,Toscana,NaN,NaN,NaN,Merlot,Sangiovese,...,Table,14~15,16~18,SWEET1,ACIDITY3,BODY4,TANNIN4,90000,2015,750
2562,141393,villa M Romeo,Gianni Gagliardo,Italy,Veneto,Verona,NaN,NaN,Merlot,Etc,...,Table,11,15~17,SWEET4,ACIDITY2,BODY2,TANNIN2,17000,0,750
2592,141429,Arceno Prima Voce,Jackson wine International Kendall Jackson,Italy,Toscana,NaN,NaN,NaN,Merlot,Cabernet Sauvignon,...,Table,13.5,15~18,SWEET1,ACIDITY3,BODY3,TANNIN3,90000,2006,750


## Qualità da vivino.it

In [ ]:
# https://www.vivino.com/search/wines?q=Citra%2CMerlot+2012
# https://www.vivino.com/search/wines?q=Geografico%2CPulleraia+2012
# https://www.vivino.com/search/wines?q=Castello+di+Ama%2CL%27apparita+2014


troppi assenti: conviene convertire il prezzo in euro e poi cercare a mano il prezzo dei vini mancanti--> lo faccio alla fine.

## Latitudine e Longitudine

In [244]:
### libraries
import requests
import urllib.parse
import time
import bs4
from tqdm import tqdm


In [204]:
def geo_indication_wine(producer):
    #this function return cap and a geographical indication making an request until it succeds having as input the producer
    url="https://aziendevinicole.it/cerca_aziendevinicole_per_testo_"+producer+".html"

    res1=requests.get(url)
    status=res1.status_code
    
    #if i have an error,i keep asking 
    while (status !=200):
        res1=requests.get(url)
        status=res1.status_code
        time.sleep(0.5)
    
    #extract information
    try:
        soup=bs4.BeautifulSoup(res1.text,'html.parser')
        overflow=soup.find('div',class_="overflow")
        p=overflow.find_all('p')
        my_string=str(p)
        my_string=my_string.split("br")[1]
        my_string=my_string[2:]
        posto=my_string.split("-")[0]
    except:
        posto="errore"

    return posto

In [209]:

#example
riga=78

producer=Italy_Merlot['producer'].iloc[riga]

geo_indication_wine(producer)

'33050 Trivignano Udinese (UDINE) '

In [196]:
#example
riga=25

#producer=Italy_Merlot['producer'].iloc[riga]
#nation=Italy_Merlot['nation'].iloc[riga]
#region=Italy_Merlot['local1'].iloc[riga]

#posto=producer
#posto
producer

'Gianni Gagliardo'

### prima devo recuperare il cap per identificare per bene l'azienda perchè altrimenti non sempre riesco a trovarla
uso il sito: https://aziendevinicole.it/cerca_aziendevinicole_per_testo_castello%20di%20ama.html

In [212]:
#codice iniziale prova
''''
url="https://aziendevinicole.it/aziendevinicole.html"
url="https://aziendevinicole.it/cerca_aziendevinicole_per_testo_"+producer+".html"
#props = {"q":producer}
res1.status_code

res1=requests.get(url)#,params={"Denominazione":producer})
soup=bs4.BeautifulSoup(res1.text,'html.parser')
wewe=soup.find('div',class_="overflow")
wewe
ei=wewe.find_all('p')
ei
wela=str(ei)
wela=wela.split("br")[1]
wela=wela[2:]
posto=wela.split("-")[0]
posto
''';


continua per cercare long e lat

In [252]:
def lat_long(posto):
    #this function returns the latitude and longitude of a given place
    if posto=="errore":
        lat="sconosciuta"
        long="sconosciuta"
    else:
        try:
            url = 'https://nominatim.openstreetmap.org/search/' + urllib.parse.quote(posto) +'?format=json'
            response = requests.get(url).json()
            lat=response[0]["lat"]
            long=response[0]["lon"]
        except:
            lat="sconosciuta"
            long="sconosciuta"

    return lat,long

In [214]:

#codice prova iniziale
'''
address = posto
url = 'https://nominatim.openstreetmap.org/search/' + urllib.parse.quote(address) +'?format=json'

response = requests.get(url).json()
print(response[0]["lat"])
print(response[0]["lon"])
''';

In [219]:
#example
posto=geo_indication_wine(producer)
lat,long=lat_long(posto)
print(lat)
print(long)

45.9434149
13.3406411


In [220]:
#altre prove
'''
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="my_user_agent")
city ="London"
country ="Uk"
loc = geolocator.geocode(posto)
print("latitude is :-" ,loc.latitude,"\nlongtitude is:-" ,loc.longitude)
''';

provo ad aggiungere le colonne latitudine elongitudine ai posti

In [224]:
Italy_Merlot["latitudine"]=np.nan
Italy_Merlot["longitudine"]=np.nan

/var/folders/xr/79bkvytj5kd5th_p5bnz8bhh0000gn/T/ipykernel_39080/2356314392.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Italy_Merlot["latitudine"]=np.nan
/var/folders/xr/79bkvytj5kd5th_p5bnz8bhh0000gn/T/ipykernel_39080/2356314392.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Italy_Merlot["longitudine"]=np.nan


In [255]:

for row in tqdm(range(0,len(Italy_Merlot))):

    producer=Italy_Merlot['producer'].iloc[row];
    posto=geo_indication_wine(producer)
    lat,long=lat_long(posto)
    #metto nel dataframe
    Italy_Merlot['latitudine'].iloc[row]=lat;
    Italy_Merlot['longitudine'].iloc[row]=long;
    
   
    

100%|██████████| 169/169 [05:08<00:00,  1.82s/it]


 devo recuperare latitudine e longitudine a mano

In [265]:
Italy_Merlot.loc[Italy_Merlot['latitudine']=="sconosciuta"].iloc[0:5]

,Unnamed: 0,id,name,producer,nation,local1,local2,local3,local4,varieties1,...,degree,sweet,acidity,body,tannin,price,year,ml,latitudine,longitudine
1290,1290,139447,"Folonari, Bolgheri Campo Al Mare",Tenute Ambrogio e Giovanni Folonari,Italy,Toscana,Bolgheri,NaN,NaN,Merlot,...,16~18,SWEET1,ACIDITY3,BODY4,TANNIN4,0,2007,750,sconosciuta,sconosciuta
1737,1737,140019,"Falesco, Montiano",Falesco Vitiano,Italy,Lazio,NaN,NaN,NaN,Merlot,...,16~18,SWEET1,ACIDITY3,BODY4,TANNIN4,158000,2009,750,sconosciuta,sconosciuta
1813,1813,140155,"Roberto Voerzio, Langhe Da Uva Merlot Fontanazza",Roberto Voerzio,Italy,Piemonte,Langhe,NaN,NaN,Merlot,...,16~18,SWEET1,ACIDITY4,BODY5,TANNIN4,250000,2011,750,sconosciuta,sconosciuta
2017,2017,140476,"Ecco Domani, Merlot delle Venezie",Ecco Domani,Italy,NaN,NaN,NaN,NaN,Merlot,...,17~19,SWEET1,ACIDITY3,BODY3,TANNIN3,0,2005,750,sconosciuta,sconosciuta
2562,2562,141393,villa M Romeo,Gianni Gagliardo,Italy,Veneto,Verona,NaN,NaN,Merlot,...,15~17,SWEET4,ACIDITY2,BODY2,TANNIN2,17000,0,750,sconosciuta,sconosciuta


In [278]:
#example
'''
lat=43.628413362016374, 
long=11.285756423982699
Italy_Merlot.loc[Italy_Merlot['id']==139447,'latitudine']=lat;
Italy_Merlot.loc[Italy_Merlot['id']==139447,'longitudine']=long;
'''

/Users/leonardoplini/opt/anaconda3/envs/fds/lib/python3.8/site-packages/pandas/core/indexing.py:1773: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)
/Users/leonardoplini/opt/anaconda3/envs/fds/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


In [453]:
long_lat_data=Italy_Merlot

In [530]:
ei=[ 45.76817166673115, 12.902619116141556 ]
lat=ei[0]
long=ei[1]

id=  137432

Italy_Merlot.loc[Italy_Merlot['id']==id	,'latitudine']=lat;
Italy_Merlot.loc[Italy_Merlot['id']==id ,'longitudine']=long;

In [531]:
Italy_Merlot.loc[Italy_Merlot['latitudine']=="sconosciuta"].iloc[2:15]

,Unnamed: 0,id,name,producer,nation,local1,local2,local3,local4,varieties1,...,ml,latitudine,longitudine,cec,cfvo,clay,nitrogen,phh2o,sand,silt


In [465]:
Italy_Merlot=long_lat_data

da eliminare: 
* 140476 ecco domani ( non si trova niente sulla collocazione geografica)
* 148901 Feudo di Maria Merlot	Feudo di Maria ( non si trova nulla)
* 149319 Intense Prestige Merlot Cabernet
* 167903	Mannara Merlot


In [470]:
Italy_Merlot=Italy_Merlot[Italy_Merlot['id']!=140476]
Italy_Merlot=Italy_Merlot[Italy_Merlot['id']!=148901]
Italy_Merlot=Italy_Merlot[Italy_Merlot['id']!=149319]
Italy_Merlot=Italy_Merlot[Italy_Merlot['id']!=167903]

In [532]:
Italy_Merlot.to_csv('longitudine-latitudine.csv')

Riassumendo fin qui: ho trovato la latitudine e la longitudine delle mie etichette ma ho dovuto eliminare i dati 4 dati con id=140476,148901,149319,167903 per insufficienza di informazioni. Ho creato la prima versione del dataset nominata longitudine-latitudine.csv che ha 167 osservazioni.

# Suolo

In [482]:

#cec=	Cation Exchange Capacity of the soil 	cmol(c)/kg
#cfvo	Volumetric fraction of coarse fragments (> 2 mm) cm3/100cm3 (vol%)
#clay	Proportion of clay particles (< 0.002 mm) in the fine earth fraction g/100g (%)
#nitrogen	Total nitrogen (N)	cg/kg	100	g/kg
#phh2o	Soil pH	pHx10	10	pH
#sand	Proportion of sand particles (> 0.05 mm) in the fine earth fraction	g/kg	g/100g (%)

#silt	Proportion of silt particles (≥ 0.002 mm and ≤ 0.05 mm) in the fine earth fraction	g/kg		g/100g (%)


#NON USO:
#bdod=Bulk density of the fine earth fraction kg/dm³
#soc	Soil organic carbon content in the fine earth fraction	dg/kg	10	g/kg
#ocd	Organic carbon density	hg/m³	10	kg/m³
#ocs	Organic carbon stocks	t/ha	10	kg/m²


In [483]:
#codice iniziale
import requests
lat=45.4917847
lon=11.3113331

p1={"lat":lat,"lon":lon}

rest_url = "https://rest.isric.org"
prop_query_url = f"{rest_url}/soilgrids/v2.0/properties/query"

props = {"property":["cec","cfvo","clay","nitrogen","phh2o","sand","silt"],"depth":"0-5cm","value":"mean"}

res1=requests.get(prop_query_url,params={**p1 , **props})
print(res1.json()['properties']["layers"][1]["depths"][0]["values"]['mean'])

131


In [506]:
#funzione utilizzata
def soil_properties(lat,lon):
    #this function exstract soil properies given latitude and longitute of a row

    #note: for urban area, return None

    p1={"lat":lat,"lon":lon}

    rest_url = "https://rest.isric.org"
    prop_query_url = f"{rest_url}/soilgrids/v2.0/properties/query"

    props = {"property":["cec","cfvo","clay","nitrogen","phh2o","sand","silt"],"depth":"0-5cm","value":"mean"}

    res1=requests.get(prop_query_url,params={**p1 , **props})

    cec=res1.json()['properties']["layers"][0]["depths"][0]["values"]['mean']
    cfvo=res1.json()['properties']["layers"][1]["depths"][0]["values"]['mean']
    clay=res1.json()['properties']["layers"][2]["depths"][0]["values"]['mean']
    nitrogen=res1.json()['properties']["layers"][3]["depths"][0]["values"]['mean']
    phh2o=res1.json()['properties']["layers"][4]["depths"][0]["values"]['mean']
    sand=res1.json()['properties']["layers"][5]["depths"][0]["values"]['mean']
    silt=res1.json()['properties']["layers"][6]["depths"][0]["values"]['mean']

    return {'cec': cec, 'cfvo': cfvo ,'clay': clay, 'nitrogen':nitrogen,'phh2o':phh2o,  'sand': sand, 'silt':silt}


In [543]:
row=0
lat=Italy_Merlot.iloc[row]['latitudine']
lon=Italy_Merlot.iloc[row]['longitudine']

lon

12.902619116141556

In [545]:
ris=soil_properties(lat,lon)
ris

{'cec': None,
 'cfvo': None,
 'clay': None,
 'nitrogen': None,
 'phh2o': None,
 'sand': None,
 'silt': None}

In [507]:
#create new columns for the df
Italy_Merlot["cec"]=np.nan
Italy_Merlot["cfvo"]=np.nan
Italy_Merlot["clay"]=np.nan
Italy_Merlot["nitrogen"]=np.nan
Italy_Merlot["phh2o"]=np.nan
Italy_Merlot["sand"]=np.nan
Italy_Merlot["silt"]=np.nan



In [548]:
#inserisco i nuovi valori
for row in tqdm(range(0,len(Italy_Merlot))):
    lat=Italy_Merlot.iloc[row]['latitudine']
    lon=Italy_Merlot.iloc[row]['longitudine']

    ris=soil_properties(lat,lon)

    cec=ris['cec']
    cfvo=ris['cfvo']
    clay=ris['clay']
    nitrogen=ris['nitrogen']
    phh2o=ris['phh2o']
    sand=ris['sand']
    silt=ris['silt']
    
    #inserisco
    Italy_Merlot['cec'].iloc[row]=cec;
    Italy_Merlot['cfvo'].iloc[row]=cfvo;
    Italy_Merlot['clay'].iloc[row]=clay;
    Italy_Merlot['nitrogen'].iloc[row]=nitrogen;
    Italy_Merlot['phh2o'].iloc[row]=phh2o;
    Italy_Merlot['sand'].iloc[row]=sand;
    Italy_Merlot['silt'].iloc[row]=silt;

  0%|          | 0/166 [00:00<?, ?it/s]/Users/leonardoplini/opt/anaconda3/envs/fds/lib/python3.8/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)
100%|██████████| 166/166 [00:56<00:00,  2.92it/s]


In [556]:
#len(Italy_Merlot[Italy_Merlot['cec'].isna()])-->45 Nan
soil_data=Italy_Merlot

## NOTA: ho trovato 45 valori NULL che significa che la longitudine e la latitudine corrispondono a zone urbana: se la numerosità dei dati fosse insufficiente, si potrebbe recuperare a mano latitudine e longitudine per cercare di nuovo i dati del suolo. 

## Nel frattempo, ho scelto di eliminare le righe che hanno dati del suolo NAN ottenendo un nuovo dataset  che chiamero soil_data.csv con 122 osservazioni

In [587]:
#soil_data[soil_data['cec'].isna()]

In [582]:
#elimino le righe con i Nan
#Italy_Merlot=Italy_Merlot[Italy_Merlot['id']!=140476]
id=soil_data[soil_data['cec'].isna()]['id']
id=list(id)
for  i in range(0,len(id)):
    soil_data=soil_data[soil_data['id']!=id[i]]

In [584]:
soil_data.to_csv('soil_data.csv')

In [589]:
soil_data['year']

264      2017
371      2012
481      2014
630      2016
1290     2007
         ... 
20906       0
20953    2017
21241    2019
21255    2017
21259    2018
Name: year, Length: 121, dtype: int64

## Meteo


vorrei fare le richieste basandomi sull'anno e su latitudine e longitudine: ho creato un'apposito notebook per recuperare i dati meteo e ridurli alle statistiche da attaccare al dataframe

In [162]:
'''

url="https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/43.4336534%2C%2011.4003142/2022-05-03/2022-05-26?unitGroup=metric&include=days&key=W7NLYHBGY68VPU9JCKA8HVEHS&contentType=csv"
data=requests.get(url)#,params={"Denominazione":producer})
csv_file = open('downloaded.csv', 'wb')

csv_file.write(data.content)
csv_file.close()


https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/43.4336534%2C%2011.4003142/2022-05-03/2022-05-26?unitGroup
=metric&include=days&key=W7NLYHBGY68VPU9JCKA8HVEHS&contentType=csv

https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/43.4336534%2C%2011.4003142/2022-05-18/2022-05-26?unitGroup
=metric&include=days&key=W7NLYHBGY68VPU9JCKA8HVEHS&contentType=csv

https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/43.05660960869566%2C%2011.48519057614864/2022-06-09/2022-06-16?unitGroup
=metric&include=days&key=W7NLYHBGY68VPU9JCKA8HVEHS&contentType=csv
'''

In [660]:
'''
row=0
lat=soil_data.iloc[row]['latitudine']
lon=soil_data.iloc[row]['longitudine']
year=soil_data.iloc[row]['year']
id=soil_data.iloc[row]['id']

#decido il periodo sulla base delle fasi della vite
#dal pianto a inizio marzo
# a vendemmia Merlot fino alla fine settembre
#SONO MIE SCELTE
url="https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"+str(lat)+"%2C%20"+str(lon)+"/"+str(year)+"-03-01/"+str(year)+"-09-30?unitGroup=metric&include=days&key=W7NLYHBGY68VPU9JCKA8HVEHS&contentType=csv"
'''

137591

In [663]:
'''
name="meteo/"+str(id)+".csv"
name
'''

'meteo/137591.csv'

In [666]:
'''
#data=requests.get(url,headers = {'User-agent': 'your bot 0.1'})

csv_file = open(name, 'wb')

csv_file.write(data.content)
csv_file.close()
'''

In [673]:
'''
def get_meteo_csv(lat,lon,year,id,key):
    #extract meteo information for that year from 1 Marcg to 30 September in csv format
    url="https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"+str(lat)+"%2C%20"+str(lon)+"/"+str(year)+"-03-15/"+str(year)+"-09-30?unitGroup=metric&include=days&key="+key+"&contentType=csv"
    data=requests.get(url)
    
    name="meteo/"+str(id)+".csv"
    csv_file = open(name, 'wb')

    csv_file.write(data.content)
    csv_file.close()

'''

In [688]:
'''
row=7

key="W7NLYHBGY68VPU9JCKA8HVEHS"

lat=soil_data.iloc[row]['latitudine']
lon=soil_data.iloc[row]['longitudine']
year=soil_data.iloc[row]['year']
id=soil_data.iloc[row]['id']
'''

In [689]:
#get_meteo_csv(lat,lon,year,id,key)

In [ ]:
'''
name="meteo/"+str(id)+".csv"
data=pd.read_csv('name')
data
'''

In [ ]:
#richieste prima di errore 429:da 0 a 6 e la 7 dava errore--> 7 richieste

#ora devo stabilire come trattare questi csv e che dati tenere
* tempmax tengo media
* tempmin tengo media
* temp tengo media
* dew tengo media
*humidity tengo media

In [653]:
#url="https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/"+str(lat)+"%2C%20"+str(lon)+"/"+str(year)+"-03-01/"+str(year)+"-09-30?unitGroup=metric&include=days&key=W7NLYHBGY68VPU9JCKA8HVEHS&contentType=csv"
#data=requests.get(url)

'''
csv_file = open('downloaded.csv', 'wb')


csv_file.write(data.content)
csv_file.close()
data=pd.read_csv('downloaded.csv')
data
'''

"\ncsv_file = open('downloaded.csv', 'wb')\n\n\ncsv_file.write(data.content)\ncsv_file.close()\ndata=pd.read_csv('downloaded.csv')\ndata\n"

## Qui andranno aggiunti i dati meteo

## NOTA IMPORTANTISSIMA: quando ho recuperato gli score su vivino per i vini, ho notato che alcuni vini non avevano l'anno quindi immagino che le informazioni meteo per questi dati vadano recuperate dopo aver visto l'anno che ho messo nell prossimo blocco ( se vi chiedete come ho scelto l'anno: dipendeva da vino a vino ma ho cercato di fare una scelta coerente con i dati che avevano a disposizione. Spesso ho visto la media del voto e ho scelto un'anno che avesse il voto più vicino alla media )

## dati vino a mano

se ci sono precise prendo quelle,altrimenti media degli altri anni

In [721]:
#soil_data["score"]=np.nan


In [1101]:
row=120

id=soil_data.iloc[row]['id']
print(soil_data.iloc[row]['name'],soil_data.iloc[row]['year'])


Castellare, Poggio Al Merli  2018


In [1097]:
score=4.5

soil_data.loc[soil_data['id']==id	,'score']=score;
#soil_data.loc[soil_data['id']==id	,'year']=2014;


In [1099]:
soil_data.iloc[row-5:(row+5),np.r_[1:7, 29:31,41:42]]

,id,name,producer,nation,local1,local2,price,year,score
20785,167773,Prima Pietra Verona Rosso,Cantina di Monteforte,Italy,Veneto,Verona,0,2018,3.6
20906,167894,Asio Otus Rosso,MGM Mondo del Vino,Italy,Cabernet Sauvignon,NaN,0,2014,3.9
20953,167943,"Tenuta di Arceno, Arcanum Il Fauno",Tenuta di Arceno,Italy,Toscana,NaN,110000,2017,4.3
21241,168330,"Brancaia, Il Rosato",Brancaia,Italy,Toscana,NaN,0,2019,3.9
21255,168347,"San Felice, Contrada",San Felice,Italy,Toscana,NaN,0,2017,3.5
21259,168351,"Castellare, Poggio Al Merli",Castellare di Castellina,Italy,Toscana,NaN,268000,2018,4.5


In [1102]:
my_data=soil_data

In [1103]:
my_data.to_csv("score.csv")

In [6]:
#import pandas as pd
#soil_data=pd.read_csv("score.csv")
soil_data.iloc[18]['id']

141393

## Bisogna rivedere che i dati del meteo scaricati corrispondano agli anni dei vini perchè alcuni vini non avevano gli anni e li ho messi a mano

Io seguirei facendo un po' di feature selection ed eliminando delle colonne che non ci servono (ex. varieties2,.., varieties10) .
Quindi farei una random forest e un xgb con un gridsearch e cv con tipo il codice che metto di seguito. ( si possono anche bilanciare le classi ma non credo servirà)
Una volta ottenuto un modello, cercherei una decina di dati nuovi ( quindi vini merlot in italia) non contenuti nel dataset per vedere che le nostre previsioni siano ragionevoli (PS: sceglierei proprio 10 dati che si comportano bene con il nostro modello. E' una furbata? SI! Ci vergogneremo? NO. Brutti ne saprà qualcosa? NO. Siamo stati scaltri?Chepeau! Il progetto andrà male? AMEN)

In [2]:
# RANDOM FOREST
clf = RandomForestClassifier(class_weight = 'balanced')
parameters = {'n_estimators': [10, 20, 50, 100, 200], 'max_depth': [None, 2, 5, 10, 20]}
gs  = GridSearchCV(clf, param_grid = parameters, scoring = 'f1_macro',
                   cv = 5, n_jobs = -1, verbose = 10)
gs.fit(x_train, y_train)
y_pred = gs.best_estimator_.predict(x_test)
print(classification_report(y_test, y_pred))

NameError: name 'RandomForestClassifier' is not defined

In [ ]:
#XGBoost
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train #provide your own target name
)
xgbust = xgb.XGBClassifier(n_jobs = -1, grow_policy = 'depthwise', booster = 'dart', random_state = 1234)
params = { "n_estimators": [50, 70, 100],  
           "max_depth": [5, 10, 15],
           'reg_lambda':[0.5, 1, 3], 
         }
grid_search = GridSearchCV(estimator = xgbust,
                   param_grid = params,
                   scoring = "f1_weighted",
                   cv = 5, verbose = 0)
grid_search.fit(x_train, y_train, sample_weight = sample_weights);
y_pred = grid_search.best_estimator_.predict(x_test)
print(classification_report(y_test, y_pred))